# Pytorch Transformation Pipeline
Will take in the ouput from Cleaning_Data.ipnyb and change into pytorch input as well as run some sanity checks. Notebooks are segmented in case we change the structure.
Tutorial: https://machinelearningmastery.com/pytorch-tutorial-develop-deep-learning-models/

```
conda create -p /projects/dagr5998/envs/ml python=3.10
conda activate /projects/dagr5998/envs/ml
conda install -c conda-forge pytorch-cpu xarray netcdf4 matplotlib ipykernel 
python -m ipykernel install --user --name ml --display-name "Python for Machine Learning"
```

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import netCDF4
from pathlib import Path
from datetime import datetime
import json
import torch

In [2]:
from helpers.defs import make_index #need to restart kernel to get these updated!!
from helpers.defs import IcepackDataset

## Config

In [5]:
RUN_NAME = "MDF_1"
window = 365 # in days, length of input series
training_vars = ["Tair", "fcondtop", 
                 'aicen_ncat1', 'aicen_ncat2', 'aicen_ncat3', 'aicen_ncat4', 'aicen_ncat5', 
                 'vicen_ncat1', 'vicen_ncat2', 'vicen_ncat3', 'vicen_ncat4', 'vicen_ncat5', 
                 'vsnon_ncat1', 'vsnon_ncat2', 'vsnon_ncat3', 'vsnon_ncat4','vsnon_ncat5']
SEED = 42
split_var = "set" #set or location
TEST_VALUES = [3,5] # choosing parameter set 3 and 5 (across all locations!!)

In [6]:
torch.manual_seed(SEED)
IN_DIR = Path(f"/projects/dagr5998/cleaned_data/01icepack_array_output/{RUN_NAME}_daily")
OUT_DIR = Path(f"/projects/dagr5998/cleaned_data/02icepack_split_output/{RUN_NAME}_split")

#### Loading and Checking Data

In [7]:
X = np.load(IN_DIR / "X.npy") # input tensor
Y = np.load(IN_DIR / "Y.npy") # output tensor
sets = np.load(IN_DIR / "sets.npy") # change parameter set index
parameterizations = np.load(IN_DIR / "parameterizations.npy") # changed parameterizations
locations = np.load(IN_DIR/"locations.npy") 
sample_len = X.shape[1]

In [8]:
with open(IN_DIR / 'config.json') as f:
    config = json.load(f)
channels = config["channels"]

In [9]:
assert list(config['shape'])==list(X.shape) 
assert(X.shape[0] == Y.shape[0] == sets.shape[0] ==parameterizations.shape[0] == locations.shape[0]) #confirms the datasets match
assert set(training_vars).issubset(channels)
input_window  = window * (24//config["aggregated hours"])

In [10]:
var_ind_list = [channels.index(v) for v in training_vars] #select relevant vars
Xt = torch.from_numpy(X[:, :, var_ind_list].copy())
Yt = torch.from_numpy(Y)

#### Splitting Testing and Training Data

In [11]:
split_col = {"set": sets, "location": locations}[split_var]
test_mask = np.isin(split_col, TEST_VALUES) #lets the split be set at top

In [12]:
train_ind = np.flatnonzero(~test_mask)
test_ind = np.flatnonzero(test_mask)

In [13]:
train_index = make_index(train_ind, sample_len, input_window)
test_index  = make_index(test_ind,  sample_len, input_window)#[run,start timestep]

In [14]:
train_runs = np.unique([r for r, b in train_index])
test_runs = np.unique([r for r, b in test_index])

In [19]:
#test_ind

### Making Dataset Class

In [14]:
mu = Xt[train_runs].mean(dim=[0,1])#calculate the mean/sd of the remaining channel
sd = Xt[train_runs].std(dim=[0,1])
print(sd.min(), sd.max())
print(training_vars[sd.argmin()], training_vars[sd.argmax()]) # highest and lowest variance channels
#plt.plot(Xt[0,:,sd.argmax()])

tensor(0.0188) tensor(18.2353)
vsnon_ncat1 fcondtop


In [16]:
train_ds = IcepackDataset(Xt, Yt, train_index, input_window,mu,sd)
test_ds  = IcepackDataset(Xt, Yt, test_index,  input_window,mu,sd)

In [17]:
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True) #shuffle the training data 
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=8, shuffle=False)

In [18]:
x,y = next(iter(test_loader))

In [19]:
print(x.mean(dim=(0,1))[:5])
print(x.std(dim=(0,1))[:5])

tensor([ 0.6039,  0.2462,  0.3365, -0.4695, -0.5896])
tensor([6.8493e-01, 1.2410e+00, 1.2768e+00, 7.5218e-01, 1.6593e-05])


In [20]:
x.shape

torch.Size([8, 365, 17])

#### Saved Output

In [21]:
Xt.shape

torch.Size([32, 16060, 17])

In [22]:
OUT_DIR.mkdir()

In [23]:
np.save(OUT_DIR / "train_index.npy", np.array(train_index, dtype=np.int32))  # (n_train, 2) [run, start]
np.save(OUT_DIR / "test_index.npy",  np.array(test_index,  dtype=np.int32))
np.save(OUT_DIR / "train_ind.npy",   train_ind)                              # run indices per side
np.save(OUT_DIR / "test_ind.npy",    test_ind)
np.save(OUT_DIR / "mu.npy",          mu.numpy())                             # (C_used,)
np.save(OUT_DIR / "sd.npy",          sd.numpy())


split_config = {
    "in_path":       str(IN_DIR),
    "training_vars": training_vars,
    "var_ind_list":  var_ind_list,
    "window_days":   window,
    "input_window":  input_window,
    "split_var":     split_var,
    "test_values":   TEST_VALUES,
    "seed":          SEED,
    "n_train":       len(train_index),
    "n_test":        len(test_index),
    "created":       datetime.now().isoformat(timespec="seconds"),
}
(OUT_DIR / "split_config.json").write_text(json.dumps(split_config, indent=2))

773